# REPA — Unified Notebook

Runs any combination of encoder and dataset by changing the two variables in the config cell.

**Supported encoders:** `meddinov3`, `dinov2`  
**Supported datasets:** `cifar10`, `chexpert`

**Required Kaggle datasets:**
- `nikiboura/meddinov3-minimal` — needed only when `ENCODER = "meddinov3"`
- `ashery/chexpert` — needed only when `DATASET = "chexpert"`

## 1. Configuration
Change only these two variables, then run all cells.

In [ ]:
# ── CHANGE THESE TWO ───────────────────────────────────────────
ENCODER = 'meddinov3'   # 'meddinov3' or 'dinov2'
DATASET = 'cifar10'     # 'cifar10'   or 'chexpert'
# ───────────────────────────────────────────────────────────────

import os

EXP_NAME     = f'repa-sits8-{ENCODER}-{DATASET}'
NUM_CLASSES  = '10' if DATASET == 'cifar10' else '2'
DATA_DIR     = f'/kaggle/working/data/{DATASET}_256'
ENC_TYPE     = 'dinov2-vit-b' if ENCODER == 'dinov2' else 'meddinov3-vit-b'
FID_REAL_DIR = DATA_DIR + ('/images/val' if DATASET == 'cifar10' else '/images/train')

CHEXPERT_ROOT  = '/kaggle/input/datasets/ashery/chexpert'
MEDDINOV3_PATH = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
MEDDINOV3_CKPT = MEDDINOV3_PATH + '/checkpoints/model.pth'

os.makedirs(DATA_DIR, exist_ok=True)

if ENCODER == 'meddinov3':
    os.environ['MEDDINOV3_PATH'] = MEDDINOV3_PATH
    os.environ['MEDDINOV3_CKPT'] = MEDDINOV3_CKPT
    print(f'MedDINOv3 exists  : {os.path.isdir(MEDDINOV3_PATH)}')
    print(f'Checkpoint exists : {os.path.isfile(MEDDINOV3_CKPT)}')

if DATASET == 'chexpert':
    print(f'CheXpert exists   : {os.path.isdir(CHEXPERT_ROOT)}')
    print(f'train.csv exists  : {os.path.isfile(os.path.join(CHEXPERT_ROOT, "train.csv"))}')

print(f'Encoder  : {ENCODER}')
print(f'Dataset  : {DATASET}')
print(f'Exp name : {EXP_NAME}')

## 2. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops pandas

## 3. Clone REPA fork

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
echo "REPA ready. Commit: $(git -C REPA rev-parse --short HEAD)"

## 4. Prepare dataset
Encodes 5000 images through the VAE and saves latents as `.npy`.

In [ ]:
import subprocess, sys

if DATASET == 'cifar10':
    cmd = [
        sys.executable, '/kaggle/working/REPA/prepare_cifar10.py',
        '--out-dir',     DATA_DIR,
        '--resolution',  '256',
        '--max-samples', '5000',
    ]
else:
    cmd = [
        sys.executable, '/kaggle/working/REPA/prepare_chexpert.py',
        '--out-dir',       DATA_DIR,
        '--chexpert-root', CHEXPERT_ROOT,
        '--resolution',    '256',
        '--max-samples',   '5000',
    ]

result = subprocess.run(cmd, text=True)
print('Exit code:', result.returncode)

## 5. Train
15000 steps, checkpoints saved to `/kaggle/working/results/`.

In [ ]:
import subprocess, os, re
from tqdm.notebook import tqdm

env = os.environ.copy()
if ENCODER == 'meddinov3':
    env['MEDDINOV3_PATH'] = MEDDINOV3_PATH
    env['MEDDINOV3_CKPT'] = MEDDINOV3_CKPT

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/kaggle/working/REPA/train.py',
    '--exp-name',            EXP_NAME,
    '--output-dir',          '/kaggle/working/results',
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         NUM_CLASSES,
    '--encoder-depth',       '8',
    '--enc-type',            ENC_TYPE,
    '--data-dir',            DATA_DIR,
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
    '--max-train-steps',     '15000',
    '--checkpointing-steps', '15000',
    '--sampling-steps',      '99999999',
]

process = subprocess.Popen(cmd, cwd='/kaggle/working/REPA', env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)

last_step = 0
pbar = tqdm(total=15000, desc=f'Training {ENCODER} x {DATASET}')
for line in process.stdout:
    m = re.search(r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)', line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
            last_step = step
pbar.close()
process.wait()
print('Training complete. Checkpoint saved.')

## 6. Check checkpoint

In [ ]:
import glob
ckpts = glob.glob('/kaggle/working/results/**/*.pt', recursive=True)
print(ckpts)

## 7. Generate samples

In [ ]:
import subprocess, os, glob

env = os.environ.copy()
if ENCODER == 'meddinov3':
    env['MEDDINOV3_PATH'] = MEDDINOV3_PATH
    env['MEDDINOV3_CKPT'] = MEDDINOV3_CKPT

ckpts = sorted(glob.glob(f'/kaggle/working/results/{EXP_NAME}/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

cmd = [
    'torchrun', '--nproc_per_node=1',
    '/kaggle/working/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          NUM_CLASSES,
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '2000',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'mse',
]

result = subprocess.run(cmd, cwd='/kaggle/working/REPA', env=env, text=True)
print('Exit code:', result.returncode)

## 8. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))
data = np.load(npz_files[-1])
imgs = data[data.files[0]]

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].astype('uint8'))
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'From: {npz_files[-1]}')

## 9. FID score

In [ ]:
import subprocess, glob, numpy as np, os
from PIL import Image
from tqdm import tqdm

subprocess.run(['pip', 'install', '-q', 'torch-fidelity'], check=True)

gen_npz = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))[-1]
print('Generated:', gen_npz)

gen_dir = '/kaggle/working/gen_images'
os.makedirs(gen_dir, exist_ok=True)
data = np.load(gen_npz)
imgs = data[data.files[0]]
for i, img in enumerate(tqdm(imgs, desc='Saving generated images')):
    Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

print(f'Real reference: {FID_REAL_DIR}')
subprocess.run([
    'fidelity',
    '--gpu', '0',
    '--fid',
    '--input1', gen_dir,
    '--input2', FID_REAL_DIR,
], text=True)